In [7]:
from credit_pipeline.data import load_dataset
from credit_pipeline import training
from credit_pipeline.evaluate import get_reject_inference_metrics
from credit_pipeline.reject_inference import *
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

Let's first load the Taiwan dataset and simulate a procedure of rejection. In this rejected set, we will assume that we do not have access to labels.

In [8]:
dataset_name = "taiwan"
seed = 0
df = load_dataset(dataset_name)
BAD_RATE = 0.25

In [9]:
# Simulate a rejected subset by randomly selecting 20% of the data, but with a different distribution of "SEX"
n_samples = int(0.2 * len(df))
n_samples_sex = {"Female": int(0.4 * n_samples), "Male": int(0.6 * n_samples)}

rejected_df = []

for sex in ["Female", "Male"]:
    df_filtered = df[df.SEX == sex]
    rejected_df.append(df_filtered.sample(n=n_samples_sex[sex], random_state=seed))

rejected_df = pd.concat(rejected_df)
accepted_df = df.drop(rejected_df.index)

In [10]:
X = accepted_df.drop(columns=["DEFAULT"])
y = accepted_df["DEFAULT"]
X_reject = rejected_df.drop(columns=["DEFAULT"])
y_reject = rejected_df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)
X_train_reject, X_test_reject, y_train_reject, y_test_reject = train_test_split(
    X_reject, y_reject, test_size=0.2, random_state=seed
)


# create concantenated training data with -1 labels for the rejected data
X_train_concat = pd.concat([X_train, X_train_reject], axis=0)
y_train_concat = pd.concat(
    [y_train, pd.Series([-1] * len(y_train_reject), index=y_train_reject.index)], axis=0
)

In [11]:
# Fit a baseline model

baseline_model = training.create_pipeline(
    X_train, y_train, LGBMClassifier(random_state=seed, verbose=-1)
)
baseline_model.fit(X_train, y_train)


model_dict = {}
model_dict["base"] = [
    baseline_model.predict_proba(X_test)[:, 1],
    baseline_model.predict_proba(X_test_reject)[:, 1],
]

In [12]:
base_estimator = LGBMClassifier(random_state=seed, verbose=-1)
reject_estimator = LGBMClassifier(random_state=seed, verbose=-1)

upward = training.create_pipeline(
    X_train,
    y_train,
    RejectUpward(base_estimator=base_estimator, reject_estimator=reject_estimator),
)
upward.fit(X_train_concat, y_train_concat)
model_dict["upward"] = [
    upward.predict_proba(X_test)[:, 1],
    upward.predict_proba(X_test_reject)[:, 1],
]

In [13]:
downward = training.create_pipeline(
    X_train,
    y_train,
    RejectDownward(base_estimator=base_estimator, reject_estimator=reject_estimator),
)
downward.fit(X_train_concat, y_train_concat)
model_dict["downward"] = [
    downward.predict_proba(X_test)[:, 1],
    downward.predict_proba(X_test_reject)[:, 1],
]

In [14]:
softcutoff = training.create_pipeline(
    X_train,
    y_train,
    RejectSoftCutoff(base_estimator=base_estimator, reject_estimator=reject_estimator),
)
softcutoff.fit(X_train_concat, y_train_concat)
model_dict["softcutoff"] = [
    softcutoff.predict_proba(X_test)[:, 1],
    softcutoff.predict_proba(X_test_reject)[:, 1],
]

In [15]:
fuzzyparcelling = training.create_pipeline(
    X_train,
    y_train,
    FuzzyParcelling(base_estimator=base_estimator, reject_estimator=reject_estimator),
)
fuzzyparcelling.fit(X_train_concat, y_train_concat)
model_dict["fuzzyparcelling"] = [
    fuzzyparcelling.predict_proba(X_test)[:, 1],
    fuzzyparcelling.predict_proba(X_test_reject)[:, 1],
]

In [16]:
for mode in ["positive", "all", "confident"]:
    extrapolation = training.create_pipeline(
        X_train,
        y_train,
        RejectExtrapolation(
            base_estimator=base_estimator, reject_estimator=reject_estimator, mode=mode
        ),
    )
    extrapolation.fit(X_train_concat, y_train_concat)
    model_dict[f"extrapolation_{mode}"] = [
        extrapolation.predict_proba(X_test)[:, 1],
        extrapolation.predict_proba(X_test_reject)[:, 1],
    ]

In [17]:
spreading = training.create_pipeline(
    X_train,
    y_train,
    RejectSpreading(base_estimator=base_estimator, reject_estimator=reject_estimator),
)
spreading.fit(X_train_concat, y_train_concat)
model_dict["spreading"] = [
    spreading.predict_proba(X_test)[:, 1],
    spreading.predict_proba(X_test_reject)[:, 1],
]

In [18]:
get_reject_inference_metrics(model_dict, y_test.values, bad_rate=BAD_RATE)

,model,approval_rate,balanced_accuracy,kickout
0,base,0.340167,0.701964,0.000000
1,upward,0.337000,0.698979,0.010692
2,downward,0.337333,0.701467,0.032817
3,softcutoff,0.338167,0.698979,0.038150
4,fuzzyparcelling,0.342833,0.697487,0.023173
5,extrapolation_positive,0.350333,0.685547,0.028757
6,extrapolation_all,0.350167,0.693009,0.049511
7,extrapolation_confident,0.329000,0.703457,0.035019
8,spreading,0.335333,0.697984,0.034222
